# 02-12c Kaggle nnU-Net PHE-only baseline 250 epochs, 19c split

Notebook nay la ban Kaggle cua `02_12` tren split moi dung chung voi `02_19c`: train PHE-only baseline bang nnU-Net framework, khong dung ICH teacher/prior.

Input/output:

```text
input  channel 0 = CT
target label     = manual PHE mask
```

Neu upload split CSV rieng thi notebook se dung split do. Neu khong co split CSV, notebook dung embedded 19c split: train 99, val 10, test 11.


In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

# Optional manual overrides for Kaggle datasets. Leave None to auto-detect data/framework.
# USER_NNUNET_REPO = Path('/kaggle/input/datasets/dovune/nnunet/nnUNet-master')
# USER_PHE_ROOT = Path('/kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
# USER_SPLIT_CSV = Path('/kaggle/input/my-split/phe_split.csv')
USER_NNUNET_REPO = None
USER_PHE_ROOT = None
USER_SPLIT_CSV = None  # keep None to use embedded 19c split

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_12c_kaggle_nnunet_phe_only_19c_split'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = NNUNET_BASE / 'nnUNet_results'

DATASET_ID = 124
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_12c_19cSplit_Kaggle'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

# Locked 19c split. Use the same IDs for every model rerun in the 19c comparison table.
VAL_SCAN_IDS = {'0013', '0014', '0051', '0060', '0061', '0078', '0113', '0116', '0160', '0167'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0080', '0084', '0095', '0096', '0115', '0119', '0130', '0138'}

for p in [OUTPUT_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

print('IS_KAGGLE            :', IS_KAGGLE)
print('WORK_ROOT            :', WORK_ROOT)
print('Output root          :', OUTPUT_ROOT)
print('Dataset              :', DATASET_NAME)
print('19c validation IDs   :', sorted(VAL_SCAN_IDS))
print('19c test IDs         :', sorted(TEST_SCAN_IDS))
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])


## 0. Locate framework and PHE data

Upload len Kaggle toi thieu 2 dataset:

- nnU-Net framework folder co `nnunetv2/__init__.py`
- PHE-SICH NIfTI folder co `SubdatasetA_NIFIT/NIFIT/set` va `label`

Notebook co auto-detect cac folder nay trong `/kaggle/input`; neu layout khac thi set `USER_NNUNET_REPO` hoac `USER_PHE_ROOT` o cell setup.


In [ ]:
def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem

def find_nnunet_repo():
    if USER_NNUNET_REPO is not None:
        p = Path(USER_NNUNET_REPO)
        if (p / 'nnunetv2' / '__init__.py').exists():
            return p
    candidates = [LOCAL_ROOT / 'nnUNet-master', LOCAL_ROOT]
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.iterdir())
    for base in candidates:
        if (base / 'nnunetv2' / '__init__.py').exists():
            return base
        if (base / 'nnUNet-master' / 'nnunetv2' / '__init__.py').exists():
            return base / 'nnUNet-master'
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    for root in search_roots:
        for init_file in root.rglob('nnunetv2/__init__.py'):
            return init_file.parents[1]
    raise FileNotFoundError('Could not find nnU-Net repo. Set USER_NNUNET_REPO manually.')

def has_nifti_pair_root(path_like):
    p = Path(path_like)
    if not (p / 'set').exists() or not (p / 'label').exists():
        return False
    image_count = len(list((p / 'set').glob('*.nii'))) + len(list((p / 'set').glob('*.nii.gz')))
    mask_count = len(list((p / 'label').glob('*.nii'))) + len(list((p / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0

def find_phe_root():
    if USER_PHE_ROOT is not None:
        p = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(p):
            return p
    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for p in candidates:
        if has_nifti_pair_root(p):
            return p
    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    nifti_roots = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            p = set_dir.parent
            if has_nifti_pair_root(p):
                nifti_roots.append(p)
    if nifti_roots:
        nifti_roots = sorted(nifti_roots, key=lambda p: (('SubdatasetA_NIFIT' not in str(p)), len(str(p))))
        return nifti_roots[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with .nii/.nii.gz files in set/label. Set USER_PHE_ROOT manually.')

def find_split_csv():
    # 12c should use the embedded 19c split by default.
    # Set USER_SPLIT_CSV explicitly only when you intentionally want to override it.
    if USER_SPLIT_CSV is not None and Path(USER_SPLIT_CSV).exists():
        return Path(USER_SPLIT_CSV)
    return None

NNUNET_REPO = find_nnunet_repo()
PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'
SPLIT_CSV = find_split_csv()

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

print('nnU-Net repo :', NNUNET_REPO)
print('PHE root     :', PHE_ROOT)
print('Image dir    :', PHE_IMAGE_DIR)
print('Mask dir     :', PHE_MASK_DIR)
print('Split CSV    :', SPLIT_CSV if SPLIT_CSV is not None else 'not found, using embedded split IDs')
print('Images       :', len(list(PHE_IMAGE_DIR.glob('*.nii*'))))
print('Masks        :', len(list(PHE_MASK_DIR.glob('*.nii*'))))


## 1. Install/import dependencies

Kaggle co the can bat Internet de cai dependency neu dataset framework chua kem moi package. Neu da cai san, cell nay chi import.

In [ ]:
AUTO_INSTALL_MISSING_DEPS = True
INSTALL_NNUNET_EDITABLE = False

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET_EDITABLE:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')
ensure_import('nnunetv2')

import SimpleITK as sitk
import nnunetv2
print('SimpleITK OK')
print('nnunetv2 import OK:', nnunetv2.__file__)

print('Using built-in trainer: nnUNetTrainer_250epochs')


## 2. Build PHE-only split manifest

Mac dinh dung locked 19c split: train 99, val 10, test 11. Notebook khong auto-detect split CSV cu trong Kaggle input; chi override split neu ban set `USER_SPLIT_CSV` ro rang o cell setup. PHE mask chi dung lam target/evaluation, input model van chi co CT.


In [ ]:
def make_case_id(scan_id):
    scan_id = str(scan_id).strip()
    if scan_id.lower().startswith('phe_'):
        return scan_id
    if scan_id.isdigit():
        return f'phe_{int(scan_id):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in scan_id).strip('_')
    return f'phe_{safe}'

def find_nii_by_scan(folder, scan_id):
    scan_id = str(scan_id).strip()
    candidates = [
        folder / f'{scan_id}.nii.gz',
        folder / f'{scan_id}.nii',
        folder / f'{int(scan_id):04d}.nii.gz' if scan_id.isdigit() else folder / f'{scan_id}.nii.gz',
        folder / f'{int(scan_id):04d}.nii' if scan_id.isdigit() else folder / f'{scan_id}.nii',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'Missing NIfTI for scan_id={scan_id} in {folder}')

def split_from_scan_id(scan_id):
    scan_id = str(scan_id).strip()
    scan_id4 = f'{int(scan_id):04d}' if scan_id.isdigit() else scan_id
    if scan_id4 in TEST_SCAN_IDS:
        return 'test'
    if scan_id4 in VAL_SCAN_IDS:
        return 'val'
    return 'train'

if SPLIT_CSV is not None:
    raw_split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
    raw_split_df.columns = [str(c).strip() for c in raw_split_df.columns]
    rows = []
    for row in raw_split_df.itertuples(index=False):
        row_dict = row._asdict()
        scan_id = row_dict.get('scan_id', None) or row_dict.get('patient_id', None)
        if scan_id is None or pd.isna(scan_id) or str(scan_id).strip() == '':
            img_path_value = row_dict.get('img_path', '')
            scan_id = strip_nii_suffix(img_path_value)
        scan_id = f'{int(str(scan_id)):04d}' if str(scan_id).isdigit() else str(scan_id).strip()
        split_value = str(row_dict.get('split', split_from_scan_id(scan_id))).strip().lower()
        img_path = Path(str(row_dict.get('img_path', '')))
        mask_path = Path(str(row_dict.get('phe_mask_path', '')))
        if not img_path.exists():
            img_path = find_nii_by_scan(PHE_IMAGE_DIR, scan_id)
        if not mask_path.exists():
            mask_path = find_nii_by_scan(PHE_MASK_DIR, scan_id)
        rows.append({'scan_id': scan_id, 'case_id': make_case_id(scan_id), 'split': split_value, 'img_path': str(img_path), 'phe_mask_path': str(mask_path)})
    split_df = pd.DataFrame(rows)
else:
    rows = []
    for img_path in sorted(PHE_IMAGE_DIR.glob('*.nii*')):
        scan_id = strip_nii_suffix(img_path)
        mask_path = find_nii_by_scan(PHE_MASK_DIR, scan_id)
        rows.append({'scan_id': scan_id, 'case_id': make_case_id(scan_id), 'split': split_from_scan_id(scan_id), 'img_path': str(img_path), 'phe_mask_path': str(mask_path)})
    split_df = pd.DataFrame(rows)

assert split_df['case_id'].is_unique, 'case_id must be unique.'
assert split_df['img_path'].map(lambda x: Path(x).exists()).all(), 'Some images are missing.'
assert split_df['phe_mask_path'].map(lambda x: Path(x).exists()).all(), 'Some masks are missing.'

split_df.to_csv(OUTPUT_ROOT / '02_12c_kaggle_phe_only_19c_split_manifest.csv', index=False)
display(split_df.groupby('split').size().rename('cases').reset_index())
display(split_df.head())


## 3. Convert PHE-only raw dataset

In [ ]:
OVERWRITE_RAW_DATASET = True

def write_binary_label(src_path: Path, reference_path: Path, dst_path: Path):
    src_path = Path(src_path)
    reference_path = Path(reference_path)
    dst_path = Path(dst_path)
    ref = sitk.ReadImage(str(reference_path))
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    if ref.GetSize() != out.GetSize():
        raise ValueError(f'Size mismatch: {reference_path} {ref.GetSize()} vs {src_path} {out.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(dst_path))

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    phe_src = Path(row.phe_mask_path)
    if row.split == 'test':
        img_dst = imagesTs / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTs / f'{case_id}.nii.gz'
    else:
        img_dst = imagesTr / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTr / f'{case_id}.nii.gz'
    shutil.copy2(img_src, img_dst)
    write_binary_label(phe_src, img_dst, seg_dst)
    records.append({'scan_id': row.scan_id, 'case_id': case_id, 'split': row.split, 'nnunet_image': str(img_dst), 'nnunet_label': str(seg_dst)})

dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_df.to_csv(OUTPUT_ROOT / '02_12c_kaggle_phe_only_19c_conversion_manifest.csv', index=False)

print('Dataset dir:', DATASET_DIR)
print('imagesTr:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
display(conversion_df.groupby('split').size().rename('cases').reset_index())


## 3b. Sanity check PHE labels

Chay cell nay truoc khi train neu `Pseudo dice` bi 0. Neu `nonzero_voxels` cua val/test > 0 thi label khong bi rong; Dice 0 o epoch dau thuong chi la model dang predict background.

In [ ]:
CHECK_LABELS = True

if CHECK_LABELS:
    qc_rows = []
    for row in conversion_df.itertuples(index=False):
        seg = sitk.ReadImage(str(row.nnunet_label))
        arr = sitk.GetArrayFromImage(seg)
        values = np.unique(arr)
        qc_rows.append({
            'case_id': row.case_id,
            'split': row.split,
            'shape_zyx': tuple(arr.shape),
            'unique_values': values.tolist(),
            'nonzero_voxels': int((arr > 0).sum()),
        })
    label_qc_df = pd.DataFrame(qc_rows)
    display(label_qc_df.groupby('split').agg(
        cases=('case_id', 'count'),
        positive_cases=('nonzero_voxels', lambda s: int((s > 0).sum())),
        min_nonzero=('nonzero_voxels', 'min'),
        median_nonzero=('nonzero_voxels', 'median'),
        max_nonzero=('nonzero_voxels', 'max'),
    ).reset_index())
    zero_cases = label_qc_df.loc[label_qc_df['nonzero_voxels'] == 0, ['case_id', 'split']]
    if len(zero_cases):
        print('WARNING: empty PHE labels found:')
        display(zero_cases)
    else:
        print('All converted PHE labels have foreground voxels.')
else:
    print('Skipped label QC.')


## 4. nnU-Net helper

In [ ]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 5. Plan and preprocess

In [ ]:
CONFIGURATION = '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run.')


## 6. Write manual fold-0 split

In [ ]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


## 7. Train PHE-only model

Mac dinh dung `nnUNetTrainer_250epochs` de bam sat baseline `02_12`. Neu muon smoke test nhanh tren Kaggle, doi `TRAINER` thanh trainer ngan hon truoc khi train.


In [ ]:
RUN_TRAIN = True
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    run_training(
        str(DATASET_ID),
        CONFIGURATION,
        str(FOLD),
        trainer_class_name=TRAINER,
        plans_identifier=PLANS,
        export_validation_probabilities=EXPORT_VAL_NPZ,
        continue_training=CONTINUE_TRAINING,
        device=device,
    )
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


## 8. Predict and evaluate test set

In [ ]:
RUN_PREDICT = True
RUN_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

print('Model dir        :', MODEL_DIR)
print('Fold dir         :', FOLD_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)

if RUN_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PREDICT=True to run inference.')

if RUN_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


## 9. Export compact metrics CSV

In [ ]:
if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON, 'r', encoding='utf-8') as f:
        summary = json.load(f)

    rows = []
    for item in summary.get('metric_per_case', []):
        metrics = item.get('metrics', {}).get('1', {})
        rows.append({
            'case_id': strip_nii_suffix(item.get('prediction_file', '')),
            'prediction_file': item.get('prediction_file', ''),
            'reference_file': item.get('reference_file', ''),
            **metrics,
        })
    metrics_df = pd.DataFrame(rows)
    metrics_csv = OUTPUT_ROOT / '02_12c_kaggle_phe_only_19c_metrics_per_case.csv'
    metrics_df.to_csv(metrics_csv, index=False)

    summary_rows = []
    numeric_cols = [c for c in metrics_df.columns if c not in {'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(metrics_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = OUTPUT_ROOT / '02_12c_kaggle_phe_only_19c_metrics_summary.csv'
    summary_df.to_csv(summary_csv, index=False)

    print('Foreground mean from nnU-Net summary:')
    print(json.dumps(summary.get('foreground_mean', {}), indent=2))
    print('Wrote:', metrics_csv)
    print('Wrote:', summary_csv)
    display(summary_df)
else:
    print('No summary yet:', SUMMARY_JSON)
    print('Run train -> predict -> evaluate first.')


## Compare with 19c-split runs

Dung notebook nay lam PHE-only nnU-Net baseline tren cung split voi `02_19c`.

- `02_12c`: CT -> PHE, 250 epochs, locked 19c split
- `02_19c`: CT + pseudo-ICH probability/distance -> PHE, locked 19c split

Khi bao cao bang 19c split, nen rerun cac model can so sanh tren dung cung `VAL_SCAN_IDS` va `TEST_SCAN_IDS` trong notebook nay.
